# Model Registry

Search, install, and manage models from the [Open Model Registry](https://github.com/GACWR/open-model-registry).

All CLI commands are also available as Python SDK functions. This notebook covers the SDK equivalents.

---
## 1. Search for Models

Search by name, description, or tags. Filter by framework and category.

In [ ]:
import openmodelstudio as oms

# Search by keyword
results = oms.registry_search("classification")
for m in results:
    print(f"{m['name']:<20} {m.get('framework',''):<10} {m.get('category',''):<16} {m.get('description','')[:50]}")

In [ ]:
# Filter by framework
results = oms.registry_search("cnn", framework="pytorch")
for m in results:
    print(f"{m['name']:<20} {m.get('description','')[:60]}")

In [ ]:
# Filter by category (empty query = all in category)
results = oms.registry_search("", category="nlp")
for m in results:
    print(f"{m['name']:<20} {m.get('description','')[:60]}")

---
## 2. Browse All Registry Models

In [ ]:
import openmodelstudio as oms

models = oms.registry_list()
print(f"{'NAME':<18} {'VERSION':<9} {'FRAMEWORK':<10} {'CATEGORY':<16} {'DESCRIPTION'}")
print("-" * 80)
for m in models:
    print(f"{m['name']:<18} {m.get('version',''):<9} {m.get('framework',''):<10} {m.get('category',''):<16} {m.get('description','')[:30]}")

---
## 3. Get Model Details

In [ ]:
import openmodelstudio as oms

info = oms.registry_info("iris-svm")
print(f"Name:         {info['name']}")
print(f"Version:      {info.get('version', '')}")
print(f"Author:       {info.get('author', '')}")
print(f"Framework:    {info.get('framework', '')}")
print(f"Category:     {info.get('category', '')}")
print(f"License:      {info.get('license', '')}")
print(f"Description:  {info.get('description', '')}")
print(f"Tags:         {', '.join(info.get('tags', []))}")
print(f"Dependencies: {', '.join(info.get('dependencies', []))}")

---
## 4. Install a Model

Downloads the model files and registers it with the platform.

In [ ]:
import openmodelstudio as oms

# Install a model
path = oms.registry_install("iris-svm")
print(f"Installed to: {path}")

# Force-reinstall an existing model
# path = oms.registry_install("iris-svm", force=True)

---
## 5. List Installed Models

In [ ]:
import openmodelstudio as oms

installed = oms.list_installed()
print(f"{'NAME':<18} {'VERSION':<9} {'FRAMEWORK':<10} {'PATH'}")
print("-" * 70)
for m in installed:
    print(f"{m['name']:<18} {m.get('version',''):<9} {m.get('framework',''):<10} {m.get('_installed_path','')[:40]}")

---
## 6. Use an Installed Model

`use_model()` loads a registry model and returns a `RegistryModel` object ready for `register_model()`. It resolves via the platform API, so it works inside workspace containers (K8s pods) without filesystem access. If the model isn’t installed yet, it auto-installs from the registry.

In [ ]:
import openmodelstudio as oms

# Load the installed registry model
iris = oms.use_model("iris-svm")
print(f"Model: {iris.name}")
print(f"Framework: {iris.framework}")
print(f"Description: {iris.description}")

---
## 7. Register and Train a Registry Model

In [ ]:
import openmodelstudio as oms

# Load and register under your own name
iris = oms.use_model("iris-svm")
handle = oms.register_model("my-iris", model=iris)
print(handle)

# Train it
job = oms.start_training(handle.model_id, wait=True)
print(f"Training: {job['status']}")

---
## 8. Uninstall a Model

Removes local files **and** unregisters from the platform, so the UI reflects the change.

In [ ]:
import openmodelstudio as oms

removed = oms.registry_uninstall("iris-svm")
print(f"Removed: {removed}")

---
## CLI Commands Reference

All of the above are also available as CLI commands:

```bash
openmodelstudio search classification
openmodelstudio search cnn --framework pytorch
openmodelstudio search "" --category nlp
openmodelstudio registry
openmodelstudio info iris-svm
openmodelstudio install iris-svm
openmodelstudio install iris-svm --force
openmodelstudio list
openmodelstudio uninstall iris-svm
openmodelstudio config
```

---
## How the Registry Works

The Open Model Registry is a GitHub repository:

```
open-model-registry/
  models/
    iris-svm/
      model.py           # train(ctx) + infer(ctx)
    mnist-cnn/
      model.py
    ...
  registry/
    index.json           # Aggregated metadata
```

Each model has a `model.py` following the `train(ctx)` / `infer(ctx)` interface. The `index.json` is read by both the CLI and the web UI to discover available models.

You can also browse and install models from the **Model Registry** page in the sidebar (Develop > Model Registry).

For full documentation, see [CLI & Model Registry docs](../docs/CLI-REGISTRY.md).